# 1. Environment Setup

This stage locates the `Task1` working directory, creates the output folders, and loads the `.env` configuration.

- Goal: keep the notebook reproducible in both the others' environment and the local development environment.
- Input: DeepSeek and DashScope settings from `Task1/.env`.
- Output: `CONFIG`, directory constants, and a safely redacted configuration preview.


In [37]:
from pathlib import Path
import shutil
from pprint import pprint
import json

from utils import (
    call_deepseek_json,
    download_image,
    extract_data_field_names,
    find_forbidden_feature_violations,
    generate_deepseek_artifact,
    build_reflection_messages,
    generate_qwen_image,
    load_env_config,
    render_plantuml_diagram,
    write_json,
    write_text,
)


def find_task1_dir(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "utils.py").is_file() and (candidate / "course_feedback_generator.ipynb").is_file():
            return candidate
    raise FileNotFoundError("Could not locate the Task1 directory containing utils.py and the notebook.")


TASK1_DIR = find_task1_dir()
ARTIFACTS_DIR = TASK1_DIR / "artifacts"
SPEC_DIR = ARTIFACTS_DIR / "spec"
DESIGN_DIR = ARTIFACTS_DIR / "design"
ASSETS_DIR = ARTIFACTS_DIR / "assets"
APP_DIR = ARTIFACTS_DIR / "app"
SEED_FEEDBACK = [
    {
        "id": 1,
        "course_name": "AI Software Engineering",
        "instructor_name": "Netzahualcoyotl Hernandez-Cruz",
        "rating": 4,
        "category": "Assessment",
        "feedback_text": "The workload is a little bit heavy.",
        "sentiment": "positive",
        "created_at": "2026-05-27T12:00:00+00:00",
    }
]
REQUIREMENTS_TEMPLATE = """flask==2.3.3
flask-cors==4.0.0
python-dotenv==1.0.0
requests==2.32.3
"""
DOCKERFILE_TEMPLATE = """FROM python:3.10-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY . .
EXPOSE 5000
CMD [\"python\", \"app.py\"]
"""
FALLBACK_HERO_IMAGE_URL = (
    "https://coresg-normal.trae.ai/api/ide/v1/text_to_image?"
    "prompt=realistic%20academic%20dashboard%20hero%20image%20for%20course%20feedback%20website%2C%20students%20reviewing%20charts%20on%20a%20modern%20laptop%2C%20clean%20university%20workspace&image_size=landscape_16_9"
)

for artifact_dir in (SPEC_DIR, DESIGN_DIR, ASSETS_DIR, APP_DIR):
    artifact_dir.mkdir(parents=True, exist_ok=True)

# Always load configuration from Task1/.env so kernel launch directories do not break secret discovery.
CONFIG = load_env_config(TASK1_DIR / ".env")
SAFE_CONFIG = {
    "deepseek_api_key_configured": bool(CONFIG.get("deepseek_api_key")),
    "dashscope_api_key_configured": bool(CONFIG.get("dashscope_api_key")),
    "deepseek_model": CONFIG["deepseek_model"],
    "qwen_image_model": CONFIG["qwen_image_model"],
    "dashscope_region": CONFIG["dashscope_region"],
    "dashscope_image_endpoint": CONFIG["dashscope_image_endpoint"],
}
pprint(SAFE_CONFIG)

{'dashscope_api_key_configured': True,
 'dashscope_image_endpoint': None,
 'dashscope_region': 'cn',
 'deepseek_api_key_configured': True,
 'deepseek_model': 'deepseek-v4-flash',
 'qwen_image_model': 'qwen-image-2.0-pro'}


# 2. Fixed Project Brief

This section defines the system boundary up front so later model outputs stay within scope.

- Must include: a single-page Flask website, fixed form fields, and fixed API routes.
- Explicitly forbidden: login, multi-role access and other complex features.
- Purpose: provide the single human-authored baseline used by Freeze Spec validation.


In [38]:
PROJECT_BRIEF = {
    "domain": "course feedback analysis",
    "app_shape": "single-page flask web app",
    "required_sections": ["hero", "form", "summary", "feedback_list"],
    "required_endpoints": ["/api/health", "/api/feedback", "/api/summary"],
    "required_api_operations": [
        {"path": "/api/health", "method": "GET"},
        {"path": "/api/feedback", "method": "GET"},
        {"path": "/api/feedback", "method": "POST"},
        {"path": "/api/summary", "method": "GET"},
    ],
    "required_fields": [
        "course_name",
        "instructor_name",
        "rating",
        "category",
        "feedback_text",
    ],
    "category_enum": ["Teaching", "Assessment", "Materials", "Pace", "Engagement"],
    "forbidden_features": [
        "login system",
        "multi-role access control",
        "runtime llm analysis",
        "multi-page navigation",
        "complex database schema",
        "rag pipelines",
        "multi-agent collaboration loops",
    ],
}

PROJECT_BRIEF

{'domain': 'course feedback analysis',
 'app_shape': 'single-page flask web app',
 'required_sections': ['hero', 'form', 'summary', 'feedback_list'],
 'required_endpoints': ['/api/health', '/api/feedback', '/api/summary'],
 'required_api_operations': [{'path': '/api/health', 'method': 'GET'},
  {'path': '/api/feedback', 'method': 'GET'},
  {'path': '/api/feedback', 'method': 'POST'},
  {'path': '/api/summary', 'method': 'GET'}],
 'required_fields': ['course_name',
  'instructor_name',
  'rating',
  'category',
  'feedback_text'],
 'category_enum': ['Teaching',
  'Assessment',
  'Materials',
  'Pace',
  'Engagement'],
 'forbidden_features': ['login system',
  'multi-role access control',
  'runtime llm analysis',
  'multi-page navigation',
  'complex database schema',
  'rag pipelines',
  'multi-agent collaboration loops']}

# 3. Inception

This phase applies an MVP-oriented inception workflow grounded in lightweight SDLC thinking, user-story-driven requirements, and strict scope control for the brief.

- It generates the problem statement, personas, requirements, and user stories.
- It also exports `acceptance_criteria.json` and `business_rules.json` as the first contract layer.
- Review focus: keep every generated item inside the course feedback website scope.


### 3.1 Prompt Builders

These helpers define the two structured DeepSeek prompts used in the inception phase.

- Input: the fixed brief and, for the second prompt, the approved problem framing.
- Output: JSON-first prompt messages for problem and requirements generation.


In [39]:
def build_problem_messages(project_brief: dict) -> list[dict[str, str]]:
    return [
        {"role": "system", "content": "Return valid JSON only."},
        {
            "role": "user",
            "content": (
                "Using this fixed brief, generate a concise problem_statement and exactly 2 personas. "
                "Each persona must include the keys name, role, goals, and frustrations. "
                "Respect all forbidden_features and do not invent extra fields. "
                f"Brief: {json.dumps(project_brief, ensure_ascii=False)}"
            ),
        },
    ]


def build_requirements_messages(project_brief: dict, problem_data: dict) -> list[dict[str, str]]:
    return [
        {"role": "system", "content": "Return valid JSON only."},
        {
            "role": "user",
            "content": (
                "Using the approved project brief and problem statement, generate JSON with the keys "
                "requirements, data_fields, page_section_mapping, endpoint_mapping, and user_stories. "
                "Return endpoint_mapping as a list of objects with path, method, and purpose. "
                "If one path supports multiple methods, include separate list items for each method. "
                "Keep every field, section, and endpoint inside the approved brief. "
                f"Brief: {json.dumps(project_brief, ensure_ascii=False)}\n"
                f"Problem data: {json.dumps(problem_data, ensure_ascii=False)}"
            ),
        },
    ]

### 3.2 Deterministic Inception Contracts

These helpers turn the generated requirements layer into acceptance criteria and business rules before the spec is frozen.


In [40]:
def build_acceptance_criteria(project_brief: dict, requirements_data: dict) -> dict:
    return {
        "ui_acceptance": {
            "sections": project_brief["required_sections"],
            "image_required": True,
            "summary_cards": [
                "total_feedback",
                "average_rating",
                "positive_ratio",
                "top_category",
            ],
        },
        "api_acceptance": {
            "required_endpoints": project_brief["required_endpoints"],
            "required_fields": project_brief["required_fields"],
        },
        "interaction_flow": [
            "Open single-page website",
            "Submit feedback with all required fields",
            "Show confirmation after submission",
            "Refresh summary metrics and feedback list",
        ],
        "traceability": {
            "page_section_mapping": requirements_data["page_section_mapping"],
            "endpoint_mapping": requirements_data["endpoint_mapping"],
        },
    }


def build_business_rules(project_brief: dict) -> dict:
    return {
        "sentiment_rules": {
            "positive": "rating >= 4",
            "neutral": "rating == 3",
            "negative": "rating <= 2",
        },
        "summary_rules": {
            "cards": [
                "total_feedback",
                "average_rating",
                "positive_ratio",
                "top_category",
            ],
            "top_category": "Most frequent feedback category",
        },
        "category_enum": project_brief["category_enum"],
    }

### 3.3 Inception Artifact Generation

These functions execute the LLM calls, normalize the returned payloads, and export the first set of Task 1 artefacts.


In [41]:
def generate_problem_artifacts(config: dict, project_brief: dict, spec_dir: Path = SPEC_DIR) -> dict:
    # Normalize model output here so every downstream phase reads the same schema.
    problem_messages = build_problem_messages(project_brief)
    payload = call_deepseek_json(
        messages=problem_messages,
        model=config["deepseek_model"],
        api_key=config["deepseek_api_key"],
        temperature=0,
        required_keys={"problem_statement", "personas"},
        max_attempts=3,
    )

    problem_statement = str(payload.get("problem_statement", "")).strip()
    personas = payload.get("personas", [])
    required_persona_keys = {"name", "role", "goals", "frustrations"}
    if not problem_statement:
        raise ValueError("problem_statement is required in the generated payload.")
    if not isinstance(personas, list) or len(personas) != 2:
        raise ValueError("The generated payload must contain exactly 2 personas.")
    normalized_personas = []
    for persona in personas:
        if not isinstance(persona, dict):
            raise ValueError("Each persona must be a JSON object.")
        normalized_persona = dict(persona)
        if "role" not in normalized_persona and isinstance(normalized_persona.get("title"), str):
            normalized_persona["role"] = normalized_persona["title"]
        missing_keys = sorted(required_persona_keys - set(normalized_persona))
        if missing_keys:
            raise ValueError(f"Each persona must include keys: {sorted(required_persona_keys)}. Missing: {missing_keys}")
        normalized_personas.append(normalized_persona)
    personas = normalized_personas

    write_text(spec_dir / "problem_statement.md", f"# Problem Statement\n\n{problem_statement}\n")
    write_json(spec_dir / "personas.json", personas)
    return {
        "problem_statement": problem_statement,
        "personas": personas,
        "generation_metadata": {
            "model": config["deepseek_model"],
            "temperature": 0,
            "prompt_count": len(problem_messages),
        },
    }


def generate_requirements_artifacts(
    # Keep the exported requirements bundle deterministic before Freeze Spec consumes it.
    config: dict,
    project_brief: dict,
    problem_data: dict,
    spec_dir: Path = SPEC_DIR,
) -> dict:
    requirements_messages = build_requirements_messages(project_brief, problem_data)
    payload = call_deepseek_json(
        messages=requirements_messages,
        model=config["deepseek_model"],
        api_key=config["deepseek_api_key"],
        temperature=0,
        required_keys={"requirements", "data_fields", "page_section_mapping", "endpoint_mapping", "user_stories"},
        max_attempts=3,
    )

    required_keys = {
        "requirements",
        "data_fields",
        "page_section_mapping",
        "endpoint_mapping",
        "user_stories",
    }
    missing = sorted(required_keys - set(payload))
    if missing:
        raise ValueError(f"Missing keys in requirements payload: {missing}")

    acceptance_criteria = build_acceptance_criteria(project_brief, payload)
    business_rules = build_business_rules(project_brief)
    requirements_value = payload.get("requirements")
    if isinstance(requirements_value, dict):
        requirements_value = dict(requirements_value)
        requirements_value.pop("forbidden_features", None)
        payload["requirements"] = requirements_value
    requirements_document = {
        key: value
        for key, value in payload.items()
        if key != "user_stories"
    }
    write_json(spec_dir / "requirements.json", requirements_document)
    write_json(spec_dir / "user_stories.json", payload["user_stories"])
    write_json(spec_dir / "acceptance_criteria.json", acceptance_criteria)
    write_json(spec_dir / "business_rules.json", business_rules)
    payload["acceptance_criteria"] = acceptance_criteria
    payload["business_rules"] = business_rules
    payload["generation_metadata"] = {
        "model": config["deepseek_model"],
        "temperature": 0,
        "prompt_count": len(requirements_messages),
    }
    return payload

### 3.4 Inception Execution

This execution cell runs the inception flow only when a valid DeepSeek key is available.


In [42]:
problem_messages = build_problem_messages(PROJECT_BRIEF)
requirements_messages = []
problem_data = None
requirements_data = None

if CONFIG["deepseek_api_key"]:
    problem_data = generate_problem_artifacts(CONFIG, PROJECT_BRIEF)
    requirements_messages = build_requirements_messages(PROJECT_BRIEF, problem_data)
    requirements_data = generate_requirements_artifacts(CONFIG, PROJECT_BRIEF, problem_data)
    print("Inception generation complete.")
else:
    print("Set DEEPSEEK_API_KEY in Task1/.env to run the inception generation cells.")

Inception generation complete.


# 4. Inception Validation

This checkpoint validates the generated inception outputs against required fields, approved sections, endpoints, and forbidden features before the spec can be frozen.

- Goal: prefer deterministic checks over subjective judgement.
- Pass condition: fields, page sections, endpoint mappings, and forbidden-feature checks all align with the fixed brief.
- Failure handling: stop before Freeze and expose the validation report for review.


### 4.1 Validation Helpers

These helper functions turn the inception artefacts into deterministic checks for fields, sections, endpoints, and forbidden scope.


In [43]:
def _collect_strings(value) -> list[str]:
    if isinstance(value, str):
        return [value]
    if isinstance(value, dict):
        collected = []
        for key, nested_value in value.items():
            collected.extend(_collect_strings(key))
            collected.extend(_collect_strings(nested_value))
        return collected
    if isinstance(value, list):
        collected = []
        for item in value:
            collected.extend(_collect_strings(item))
        return collected
    return []


def _strip_forbidden_feature_metadata(value):
    if isinstance(value, dict):
        return {
            key: _strip_forbidden_feature_metadata(nested_value)
            for key, nested_value in value.items()
            if key != "forbidden_features"
        }
    if isinstance(value, list):
        return [_strip_forbidden_feature_metadata(item) for item in value]
    return value


def validate_required_fields(spec: dict) -> dict:
    expected = set(PROJECT_BRIEF["required_fields"])
    actual = set(extract_data_field_names(spec.get("data_fields", [])))
    return {
        "ok": actual == expected,
        "expected": sorted(expected),
        "actual": sorted(actual),
    }


def validate_forbidden_features(problem_data: dict, requirements_data: dict) -> dict:
    # Treat negative scope statements as compliant constraints, not feature violations.
    # Only flag affirmative scope violations, not negative constraints such as "No login system".
    requirements_scope = _strip_forbidden_feature_metadata(requirements_data.get("requirements", []))
    text_blocks = []
    text_blocks.extend(_collect_strings(problem_data.get("problem_statement", "")))
    text_blocks.extend(_collect_strings(_strip_forbidden_feature_metadata(problem_data.get("constraints", []))))
    text_blocks.extend(_collect_strings(requirements_scope))
    text_blocks.extend(_collect_strings(_strip_forbidden_feature_metadata(requirements_data.get("user_stories", []))))
    violations = find_forbidden_feature_violations(
        text_blocks,
        PROJECT_BRIEF["forbidden_features"],
    )
    return {"ok": not violations, "violations": violations}


def validate_section_mapping(spec: dict) -> dict:
    expected = set(PROJECT_BRIEF["required_sections"])
    actual = set(_collect_strings(spec.get("page_section_mapping", {}))) & expected
    return {
        "ok": actual == expected,
        "expected": sorted(expected),
        "actual": sorted(actual),
    }


def _extract_endpoint_operations(endpoint_mapping) -> list[str]:
    if isinstance(endpoint_mapping, list):
        operations = []
        for item in endpoint_mapping:
            if not isinstance(item, dict):
                continue
            path = item.get("path")
            method = item.get("method")
            if isinstance(path, str) and isinstance(method, str):
                operations.append(f"{method.upper()} {path}")
        return sorted(set(operations))

    if isinstance(endpoint_mapping, dict):
        operations = []
        for path, details in endpoint_mapping.items():
            if not isinstance(path, str) or not isinstance(details, dict):
                continue
            method = details.get("method")
            if isinstance(method, str):
                operations.append(f"{method.upper()} {path}")
        return sorted(set(operations))

    return []


def validate_endpoint_mapping(spec: dict) -> dict:
    expected = sorted(
        f"{item['method'].upper()} {item['path']}"
        for item in PROJECT_BRIEF["required_api_operations"]
    )
    actual = _extract_endpoint_operations(spec.get("endpoint_mapping", {}))
    return {
        "ok": actual == expected,
        "expected": expected,
        "actual": actual,
    }

### 4.2 Validation Report Assembly

This section composes checkpoint reports and raises a blocking error before Freeze Spec when the inception output is inconsistent.


In [44]:
def build_checkpoint_report(problem_data: dict, requirements_data: dict, checks: dict) -> dict:
    checkpoint_a = {
        "approved": bool(problem_data.get("problem_statement") and problem_data.get("personas")),
        "artifacts": ["problem_statement.md", "personas.json"],
    }
    checkpoint_b = {
        "approved": checkpoint_a["approved"]
        and bool(requirements_data.get("requirements"))
        and bool(requirements_data.get("user_stories"))
        and bool(requirements_data.get("acceptance_criteria"))
        and bool(requirements_data.get("business_rules"))
        and all(result["ok"] for result in checks.values()),
        "artifacts": [
            "requirements.json",
            "user_stories.json",
            "acceptance_criteria.json",
            "business_rules.json",
        ],
    }
    return {
        "Checkpoint A": checkpoint_a,
        "Checkpoint B": checkpoint_b,
    }


def build_validation_report(problem_data: dict, requirements_data: dict) -> dict:
    checks = {
        "required_fields": validate_required_fields(requirements_data),
        "forbidden_features": validate_forbidden_features(problem_data, requirements_data),
        "page_section_mapping": validate_section_mapping(requirements_data),
        "endpoint_mapping": validate_endpoint_mapping(requirements_data),
    }
    checkpoints = build_checkpoint_report(problem_data, requirements_data, checks)
    return {
        "approved": all(result["ok"] for result in checks.values())
        and all(item["approved"] for item in checkpoints.values()),
        "checkpoints": checkpoints,
        "checks": checks,
    }


def assert_inception_is_valid(problem_data: dict, requirements_data: dict) -> dict:
    report = build_validation_report(problem_data, requirements_data)
    if not report["approved"]:
        raise ValueError(json.dumps(report, indent=2, ensure_ascii=False))
    return report

# 5. Freeze Spec

This phase turns the approved inception outputs into a stable and auditable specification snapshot so that later construction steps read from a frozen contract baseline instead of free-form text.

- `Checkpoint A`: problem statement and personas.
- `Checkpoint B`: requirements, user stories, acceptance criteria, and business rules.
- Output: `frozen_inception_spec.json` becomes the single source of truth for downstream generation.


In [45]:
def freeze_inception_spec(
    project_brief: dict,
    problem_data: dict,
    requirements_data: dict,
    spec_dir: Path = SPEC_DIR,
) -> tuple[dict, dict]:
    validation_report = assert_inception_is_valid(problem_data, requirements_data)
    acceptance_criteria = requirements_data["acceptance_criteria"]
    business_rules = requirements_data["business_rules"]
    frozen_spec = {
        "project_brief": project_brief,
        "problem_statement": problem_data["problem_statement"],
        "personas": problem_data["personas"],
        "requirements": requirements_data["requirements"],
        "user_stories": requirements_data["user_stories"],
        "data_fields": requirements_data["data_fields"],
        "page_section_mapping": requirements_data["page_section_mapping"],
        "endpoint_mapping": requirements_data["endpoint_mapping"],
        "acceptance_criteria": acceptance_criteria,
        "business_rules": business_rules,
        "validation_report": validation_report,
        "generation_metadata": {
            "problem": problem_data.get("generation_metadata", {}),
            "requirements": requirements_data.get("generation_metadata", {}),
        },
    }
    write_json(spec_dir / "frozen_inception_spec.json", frozen_spec)
    return frozen_spec, validation_report


frozen_spec = None
validation_report = None

if problem_data and requirements_data:
    frozen_spec, validation_report = freeze_inception_spec(
        PROJECT_BRIEF,
        problem_data,
        requirements_data,
    )
    pprint(validation_report)
else:
    print("Run the inception generation cell first to export frozen_inception_spec.json.")

{'approved': True,
 'checkpoints': {'Checkpoint A': {'approved': True,
                                  'artifacts': ['problem_statement.md',
                                                'personas.json']},
                 'Checkpoint B': {'approved': True,
                                  'artifacts': ['requirements.json',
                                                'user_stories.json',
                                                'acceptance_criteria.json',
                                                'business_rules.json']}},
 'checks': {'endpoint_mapping': {'actual': ['GET /api/feedback',
                                            'GET /api/health',
                                            'GET /api/summary',
                                            'POST /api/feedback'],
                                 'expected': ['GET /api/feedback',
                                              'GET /api/health',
                                              'GET /api/sum

# 6. Construction

This phase uses an AI-DLC construction pipeline to translate the frozen specification into contract-constrained API, UI, UML, and prompt artefacts with explicit state hand-offs.

- LangGraph is used as a lightweight orchestration layer.
- Each node keeps a narrow responsibility and writes traceable outputs back to `artifacts/spec`.


### 6.1 State And Contract Builders

This section defines the shared LangGraph state and converts the frozen spec into backend, frontend, and UML contracts.


In [46]:
from typing import TypedDict

from langgraph.graph import END, START, StateGraph


class BuildState(TypedDict, total=False):
    frozen_spec: dict
    backend_contract: dict
    frontend_contract: dict
    uml_contract: dict
    api_spec: dict
    ui_spec: dict
    uml_source: dict
    backend_code: str
    frontend_code: str
    image_prompt: str
    rendered_files: dict
    validation_report: dict


def build_backend_contract(frozen_spec: dict) -> dict:
    project_brief = frozen_spec["project_brief"]
    business_rules = frozen_spec["business_rules"]
    return {
        "routes": project_brief["required_endpoints"],
        "required_fields": project_brief["required_fields"],
        "summary_cards": business_rules["summary_rules"]["cards"],
        "index_file": "index.html",
        "data_file": "seed_feedback.json",
        "data_path_symbol": "DATA_PATH",
        "runtime_options_symbol": "get_runtime_options",
        "error_response_key": "error",
        "persistence_mode": "file_backed_per_request",
        "feedback_metadata_fields": ["id", "sentiment", "created_at"],
        "runtime_defaults": {"host": "0.0.0.0", "host_env": "FLASK_HOST", "port_env": "PORT", "port": 5000, "debug_env": "FLASK_DEBUG", "debug": False},
        "success_message": "Feedback submitted successfully.",
        "missing_fields_prefix": "Missing required fields:",
        "feedback_id_mode": "incrementing_integer",
        "category_counts_mode": "observed_only",
        "root_route_mode": "serve_from_app_directory",
        "runtime_options_shape": "dict",
        "image_file": "generated_hero.png",
        "health_app_name": "course-feedback-analysis",
        "health_status": "ok",
        "summary_ratio_scale": "percentage_0_100",
        "response_shapes": {
            "health": ["status", "app"],
            "feedback_list": ["feedback"],
            "feedback_post": ["message", "feedback"],
            "summary": [
                "total_feedback",
                "average_rating",
                "positive_ratio",
                "top_category",
                "category_counts",
            ],
        },
    }


def build_frontend_contract(frozen_spec: dict) -> dict:
    project_brief = frozen_spec["project_brief"]
    business_rules = frozen_spec["business_rules"]
    return {
        "sections": project_brief["required_sections"],
        "field_names": project_brief["required_fields"],
        "category_enum": business_rules["category_enum"],
        "summary_cards": business_rules["summary_rules"]["cards"],
        "image_file": "generated_hero.png",
        "page_title": "Course Feedback Analysis System",
        "required_dom_ids": ["hero", "feedback-form", "summary-section", "feedback-list"],
        "feedback_collection_reference": "feedbackData.feedback",
        "feedback_iteration_target": "feedbackData.feedback",
    }


def build_uml_contract(frozen_spec: dict) -> dict:
    actors = []
    for persona in frozen_spec["personas"]:
        actor_name = persona.get("role") or persona.get("title")
        if not actor_name:
            raise ValueError("Each persona must provide role or title for UML actors.")
        actors.append(actor_name)
    return {
        "actors": actors,
        "user_stories": frozen_spec["user_stories"],
        "workflow_steps": [
            "Open single-page website",
            "Review summary metrics",
            "Complete feedback form",
            "Submit feedback via Flask API",
            "Persist feedback entry",
            "Refresh summary and feedback list",
        ],
    }

### 6.2 UML Source Generation

This section builds UML either through a constrained LLM call or through a deterministic fallback when validation fails.


In [47]:
def build_deterministic_uml_source(uml_contract: dict) -> dict:
    actor_lines = []
    actor_links = []
    for index, actor_name in enumerate(uml_contract["actors"], start=1):
        alias = f"A{index}"
        actor_lines.append(f'actor "{actor_name}" as {alias}')
        actor_links.extend(
            [
                f"{alias} --> UC1",
                f"{alias} --> UC2",
                f"{alias} --> UC3",
            ]
        )
    use_case_diagram = "\n".join(
        [
            "@startuml",
            "left to right direction",
            *actor_lines,
            'rectangle "Course Feedback Generator" {',
            '  usecase "Submit Feedback" as UC1',
            '  usecase "View Summary Metrics" as UC2',
            '  usecase "Review Recent Feedback" as UC3',
            "}",
            *actor_links,
            "@enduml",
        ]
    )
    activity_lines = [
        "@startuml",
        "start",
    ]
    for step in uml_contract["workflow_steps"]:
        activity_lines.append(f":{step};")
    activity_lines.extend(["stop", "@enduml"])
    return {
        "use_case": use_case_diagram,
        "activity": "\n".join(activity_lines),
    }


def generate_uml_source_from_contract(config: dict, uml_contract: dict) -> dict:
    prompt = {
        "actors": uml_contract["actors"],
        "user_stories": uml_contract["user_stories"],
        "workflow_steps": uml_contract["workflow_steps"],
        "rules": [
            "Return valid JSON only.",
            "Use PlantUML syntax.",
            "Return keys use_case and activity.",
            "Each UML string must start with @startuml and end with @enduml.",
            "The use_case value must contain a complete PlantUML use case diagram.",
            "The activity value must contain a complete PlantUML activity diagram.",
            "Do not include out-of-scope features.",
        ],
    }
    try:
        payload = call_deepseek_json(
            messages=[
                {
                    "role": "system",
                    "content": "Return JSON with keys use_case and activity. Each value must be complete PlantUML source code enclosed by @startuml and @enduml.",
                },
                {
                    "role": "user",
                    "content": json.dumps(prompt, ensure_ascii=False),
                },
            ],
            model=config["deepseek_model"],
            api_key=config["deepseek_api_key"],
            temperature=0,
            required_keys={"use_case", "activity"},
            max_attempts=3,
        )
    except Exception:
        return build_deterministic_uml_source(uml_contract)
    payload_findings = validate_uml_source(payload, uml_contract)
    if payload_findings:
        return build_deterministic_uml_source(uml_contract)
    return {
        "use_case": payload["use_case"],
        "activity": payload["activity"],
    }

### 6.3 Design Output Generation

These functions assemble API and UI specs and then ask the LLM to generate backend and frontend source code under contract rules.


In [48]:
def design_system(state: BuildState) -> BuildState:
    spec = state["frozen_spec"]
    project_brief = spec["project_brief"]
    backend_contract = build_backend_contract(spec)
    frontend_contract = build_frontend_contract(spec)
    uml_contract = build_uml_contract(spec)
    return {
        "backend_contract": backend_contract,
        "frontend_contract": frontend_contract,
        "uml_contract": uml_contract,
        "api_spec": {
            "routes": [
                {
                    "method": "GET",
                    "path": "/api/health",
                    "purpose": "Return a lightweight health response for smoke checks.",
                },
                {
                    "method": "GET",
                    "path": "/api/feedback",
                    "purpose": "Return stored feedback entries for the dashboard list.",
                },
                {
                    "method": "POST",
                    "path": "/api/feedback",
                    "purpose": "Persist a feedback entry from the submission form.",
                },
                {
                    "method": "GET",
                    "path": "/api/summary",
                    "purpose": "Return summary metrics for the dashboard cards.",
                },
            ],
            "required_endpoints": backend_contract["routes"],
            "required_fields": backend_contract["required_fields"],
        },
        "ui_spec": {
            "sections": frontend_contract["sections"],
            "fields": project_brief["required_fields"],
            "category_enum": frontend_contract["category_enum"],
            "summary_cards": frontend_contract["summary_cards"],
            "image_file": frontend_contract["image_file"],
        },
    }



def generate_backend_code(config: dict, frozen_spec: dict, backend_contract: dict) -> str:
    # Keep prompt rules explicit so the generated backend stays inside the frozen contract.
    request_payload = {
        "frozen_spec": frozen_spec,
        "backend_contract": backend_contract,
        "rules": [
            "Return Python source code only.",
            "Use Flask.",
            "Implement only the required routes.",
            "Serve the generated index.html file for the root route.",
            'Serve the root route from the app directory with send_from_directory(app.root_path, "index.html") or equivalent, not app.send_static_file.',
            "Do not embed HTML templates inside app.py.",
            "Persist feedback in seed_feedback.json inside the app directory.",
            "Expose DATA_PATH as a pathlib.Path constant.",
            "Define get_runtime_options() to read host, port, and debug settings.",
            "Read the runtime host from FLASK_HOST and debug mode from FLASK_DEBUG.",
            "Parse FLASK_DEBUG flexibly so true, 1, and yes all enable debug mode.",
            "Read and write feedback directly from seed_feedback.json for each request; do not keep a module-level feedback cache.",
            "get_runtime_options() must return a dict with keys host, port, and debug.",
            "Default runtime options to host 0.0.0.0, PORT 5000, and debug False.",
            "Serve generated_hero.png as a static asset.",
            'Return GET /api/health as {"status": "ok", "app": "course-feedback-analysis"}.',
            "Return POST /api/feedback payloads with keys message and feedback.",
            "Return the exact success message Feedback submitted successfully.",
            "Validation failures must return JSON with an error key.",
            "Format missing required-field errors as Missing required fields: ... with alphabetical field ordering.",
            "Persist each feedback entry with an incrementing integer id plus sentiment and created_at fields.",
            "Return category_counts with observed categories only.",
            "Accept browser-style string ratings by converting them to integers before backend validation.",
            "Return positive_ratio as a percentage between 0 and 100.",
            "Do not add authentication, databases, or runtime LLM analysis.",
        ],
    }
    return generate_deepseek_artifact(
        system_instruction=(
            "Return JSON with a single key named artifact. "
            "The artifact value must be valid Python source code only."
        ),
        user_instruction=json.dumps(request_payload, ensure_ascii=False),
        model=config["deepseek_model"],
        api_key=config["deepseek_api_key"],
        temperature=0,
        max_attempts=3,
    )


def generate_frontend_code(config: dict, frozen_spec: dict, frontend_contract: dict) -> str:
    request_payload = {
        "frozen_spec": frozen_spec,
        "frontend_contract": frontend_contract,
        "rules": [
            "Return HTML only.",
            "Keep a single-page layout.",
            "Reference generated_hero.png.",
            "Use only the required sections and fields.",
            "Use the page title Course Feedback Analysis System.",
            "Use DOM ids hero, feedback-form, summary-section, and feedback-list.",
            "Read the feedback list from feedbackData.feedback returned by GET /api/feedback.",
            "Store the full API response in feedbackData and iterate only over feedbackData.feedback, never over the raw response object.",
            "Convert the rating field to a number before POST /api/feedback.",
            "Treat positive_ratio from /api/summary as a percentage value already scaled to 0-100.",
        ],
    }
    return generate_deepseek_artifact(
        system_instruction=(
            "Return JSON with a single key named artifact. "
            "The artifact value must be valid HTML only."
        ),
        user_instruction=json.dumps(request_payload, ensure_ascii=False),
        model=config["deepseek_model"],
        api_key=config["deepseek_api_key"],
        temperature=0,
        max_attempts=3,
    )

### 6.4 Deterministic Validation And Reflection

These helpers validate generated artefacts first and then allow only narrow, contract-bounded reflection when findings exist.


In [49]:
def validate_backend_artifact(code_text: str, backend_contract: dict) -> list[str]:
    # These checks act as deterministic guardrails before any reflection step runs.
    findings = []
    for route in backend_contract["routes"]:
        if route not in code_text:
            findings.append(f"Missing route reference: {route}")
    for field in backend_contract["required_fields"]:
        if field not in code_text:
            findings.append(f"Missing field reference: {field}")
    if "render_template_string" in code_text:
        findings.append("Backend must not embed HTML templates inside app.py.")
    if backend_contract["index_file"] not in code_text:
        findings.append(f"Missing external frontend file reference: {backend_contract['index_file']}")
    if backend_contract.get("image_file") and backend_contract["image_file"] not in code_text:
        findings.append("Backend must serve generated hero image from the app directory.")
    if "send_static_file" in code_text:
        findings.append("Backend root route must serve index.html from the app directory, not Flask static.")
    if backend_contract["data_file"] not in code_text:
        findings.append(f"Missing persistent feedback file reference: {backend_contract['data_file']}")
    if backend_contract.get("data_path_symbol") and backend_contract["data_path_symbol"] not in code_text:
        findings.append(f"Missing DATA_PATH symbol: {backend_contract['data_path_symbol']}")
    if backend_contract.get("persistence_mode") == "file_backed_per_request" and "feedback_list =" in code_text:
        findings.append("Backend must read feedback from seed_feedback.json per request, not from a module-level cache.")
    if backend_contract.get("runtime_options_symbol") and backend_contract["runtime_options_symbol"] not in code_text:
        findings.append(f"Missing runtime options helper: {backend_contract['runtime_options_symbol']}")
    if backend_contract.get("runtime_options_shape") == "dict":
        if "host" not in code_text or "port" not in code_text or "debug" not in code_text:
            findings.append("Runtime options helper must return a dict with host, port, and debug keys.")
    if backend_contract.get("error_response_key") and backend_contract["error_response_key"] not in code_text:
        findings.append(f"Missing error response key: {backend_contract['error_response_key']}")
    if backend_contract.get("health_app_name") and backend_contract["health_app_name"] not in code_text:
        findings.append(f"Missing health app name: {backend_contract['health_app_name']}")
    if backend_contract.get("health_status") and backend_contract["health_status"] not in code_text:
        findings.append(f"Missing health status value: {backend_contract['health_status']}")
    if backend_contract.get("health_status") == "ok" and ("'status': 'healthy'" in code_text or "\"status\": \"healthy\"" in code_text):
        findings.append("Health endpoint must return status ok.")
    for metadata_field in backend_contract.get("feedback_metadata_fields", []):
        if metadata_field not in code_text:
            findings.append(f"Missing feedback metadata field: {metadata_field}")
    runtime_defaults = backend_contract.get("runtime_defaults", {})
    if runtime_defaults.get("host") and runtime_defaults["host"] not in code_text:
        findings.append(f"Missing runtime default host: {runtime_defaults['host']}")
    if runtime_defaults.get("host_env") and runtime_defaults["host_env"] not in code_text:
        findings.append(f"Missing runtime host environment variable: {runtime_defaults['host_env']}")
    if runtime_defaults.get("port_env") and runtime_defaults["port_env"] not in code_text:
        findings.append(f"Missing runtime port environment variable: {runtime_defaults['port_env']}")
    if runtime_defaults.get("debug_env") and runtime_defaults["debug_env"] not in code_text:
        findings.append(f"Missing runtime debug environment variable: {runtime_defaults['debug_env']}")
    if runtime_defaults.get("debug") is False and "False" not in code_text:
        findings.append("Missing runtime debug default: False")
    strict_debug_patterns = [
        "os.environ.get('FLASK_DEBUG', 'false').lower() == 'true'",
        "os.environ.get(\"FLASK_DEBUG\", \"false\").lower() == \"true\"",
        "os.environ.get('FLASK_DEBUG', 'False').lower() == 'true'",
        "os.environ.get(\"FLASK_DEBUG\", \"False\").lower() == \"true\"",
    ]
    if any(pattern in code_text for pattern in strict_debug_patterns):
        findings.append("Runtime debug parsing must accept true/1/yes values, not only the exact string true.")
    if backend_contract.get("success_message") and backend_contract["success_message"] not in code_text:
        findings.append(f"Missing success message: {backend_contract['success_message']}")
    if (('rating = data.get("rating")' in code_text) or ("rating = data.get('rating')" in code_text)) and "isinstance(rating, int)" in code_text and 'data["rating"] = int(data["rating"])' not in code_text and "data['rating'] = int(data['rating'])" not in code_text:
        findings.append("Backend must normalise browser-style string ratings before validating the rating range.")
    if backend_contract.get("missing_fields_prefix") and backend_contract["missing_fields_prefix"] not in code_text:
        findings.append(f"Missing required-fields error prefix: {backend_contract['missing_fields_prefix']}")
    bad_missing_field_patterns = [
        "f'Missing required fields: {missing}'",
        'f"Missing required fields: {missing}"',
        "f'Missing required fields: {missing_sorted}'",
        'f"Missing required fields: {missing_sorted}"',
    ]
    if any(pattern in code_text for pattern in bad_missing_field_patterns):
        findings.append("Missing required-fields error must join field names into plain text, not a Python list representation.")
    if backend_contract.get("feedback_id_mode") == "incrementing_integer" and ("random" in code_text or "choices(" in code_text):
        findings.append("Feedback ids must be incrementing integers, not random strings.")
    if backend_contract.get("category_counts_mode") == "observed_only" and "{cat: 0 for cat in CATEGORY_ENUM}" in code_text:
        findings.append("category_counts must omit zero-count categories.")
    if backend_contract.get("summary_ratio_scale") == "percentage_0_100" and "* 100" not in code_text and "*100" not in code_text:
        findings.append("Missing percentage positive_ratio implementation.")
    return findings


def validate_frontend_artifact(html_text: str, frontend_contract: dict) -> list[str]:
    findings = []
    for section in frontend_contract["sections"]:
        aliases = {section, section.replace("_", "-")}
        if not any(alias in html_text for alias in aliases):
            findings.append(f"Missing section reference: {section}")
    for dom_id in frontend_contract.get("required_dom_ids", []):
        if f'id="{dom_id}"' not in html_text:
            findings.append(f"Missing DOM id: {dom_id}")
    if frontend_contract["image_file"] not in html_text:
        findings.append("Missing generated hero image reference.")
    page_title = frontend_contract.get("page_title")
    if page_title and f"<title>{page_title}</title>" not in html_text:
        findings.append(f"Missing page title: {page_title}")
    feedback_collection_reference = frontend_contract.get("feedback_collection_reference")
    if feedback_collection_reference and feedback_collection_reference not in html_text:
        findings.append(
            "Missing feedback collection reference: feedbackData.feedback"
        )
    if "feedbackData.feedback = data" in html_text or "data.forEach" in html_text or "data.length" in html_text:
        findings.append("Frontend must iterate over feedbackData.feedback, not the raw API response object.")
    numeric_rating_patterns = [
        "key === 'rating' ? Number(value) : value",
        "key === \"rating\" ? Number(value) : value",
        "key === 'rating'",
        "key === \"rating\"",
        "payload.rating = Number(payload.rating)",
        "parseInt(payload.rating",
    ]
    has_numeric_rating_conversion = (
        (("key === 'rating'" in html_text or "key === \"rating\"" in html_text) and "Number(value)" in html_text)
        or "payload.rating = Number(payload.rating)" in html_text
        or "parseInt(payload.rating" in html_text
    )
    if "FormData(" in html_text and ("payload[key] = value" in html_text or "payload[key]=value" in html_text) and not has_numeric_rating_conversion:
        findings.append("Frontend must convert the rating field to a number before POST /api/feedback.")
    return findings


def validate_uml_source(uml_source: dict, uml_contract: dict) -> list[str]:
    findings = []
    if not isinstance(uml_source, dict):
        return ["UML source must be a JSON object."]
    for diagram_key in ("use_case", "activity"):
        diagram_text = uml_source.get(diagram_key)
        if not isinstance(diagram_text, str) or not diagram_text.strip():
            findings.append(f"Missing UML diagram text for {diagram_key}.")
            continue
        stripped_text = diagram_text.strip()
        if not stripped_text.startswith("@startuml") or not stripped_text.endswith("@enduml"):
            findings.append("Each UML string must start with @startuml and end with @enduml.")
    use_case_text = uml_source.get("use_case", "")
    activity_text = uml_source.get("activity", "")
    for actor_name in uml_contract["actors"]:
        if actor_name not in use_case_text:
            findings.append(f"Missing UML actor reference: {actor_name}")
    for step in uml_contract["workflow_steps"]:
        if step not in activity_text:
            findings.append(f"Missing UML workflow step: {step}")
    return findings


def run_narrow_reflection(
    config: dict,
    artifact_type: str,
    artifact_text: str,
    contract: dict,
    findings: list[str],
) -> str:
    if not findings:
        return artifact_text
    response = call_deepseek_json(
        messages=build_reflection_messages(
            artifact_type=artifact_type,
            artifact_text=artifact_text,
            contract=contract,
            findings=findings,
        ),
        model=config["deepseek_model"],
        api_key=config["deepseek_api_key"],
        temperature=0,
        required_keys={"artifact"},
        max_attempts=3,
    )
    return response["artifact"]

### 6.5 Graph Execution

This execution cell compiles the lightweight LangGraph pipeline and runs construction only after Freeze Spec has produced an approved baseline.


In [50]:
def generate_artifacts(state: BuildState) -> BuildState:
    spec = state["frozen_spec"]
    problem_statement = spec["problem_statement"]
    image_prompt = (
        "A realistic academic dashboard hero image for a course feedback analysis website, "
        "students reviewing charts on a modern laptop, clean university workspace, soft blue palette, "
        f"professional website illustration. Context: {problem_statement}"
    )
    backend_code = generate_backend_code(CONFIG, state["frozen_spec"], state["backend_contract"])
    frontend_code = generate_frontend_code(CONFIG, state["frozen_spec"], state["frontend_contract"])
    uml_source = generate_uml_source_from_contract(CONFIG, state["uml_contract"])
    uml_findings = validate_uml_source(uml_source, state["uml_contract"])
    if uml_findings:
        uml_source = build_deterministic_uml_source(state["uml_contract"])
    uml_findings = validate_uml_source(uml_source, state["uml_contract"])

    backend_findings = validate_backend_artifact(backend_code, state["backend_contract"])
    frontend_findings = validate_frontend_artifact(frontend_code, state["frontend_contract"])

    backend_code = run_narrow_reflection(CONFIG, "backend", backend_code, state["backend_contract"], backend_findings)
    frontend_code = run_narrow_reflection(CONFIG, "frontend", frontend_code, state["frontend_contract"], frontend_findings)

    final_backend_findings = validate_backend_artifact(backend_code, state["backend_contract"])
    final_frontend_findings = validate_frontend_artifact(frontend_code, state["frontend_contract"])
    validation_report = {
        "approved": not final_backend_findings and not final_frontend_findings and not uml_findings,
        "backend_findings": final_backend_findings,
        "frontend_findings": final_frontend_findings,
        "uml_findings": uml_findings,
    }
    return {
        "backend_code": backend_code,
        "frontend_code": frontend_code,
        "uml_source": uml_source,
        "image_prompt": image_prompt,
        "validation_report": validation_report,
    }


graph = StateGraph(BuildState)
graph.add_node("design_system", design_system)
graph.add_node("generate_artifacts", generate_artifacts)
graph.add_edge(START, "design_system")
graph.add_edge("design_system", "generate_artifacts")
graph.add_edge("generate_artifacts", END)
compiled_graph = graph.compile()

build_state = None
if frozen_spec:
    build_state = compiled_graph.invoke({"frozen_spec": frozen_spec})
    pprint({key: value for key, value in build_state.items() if key != "uml_source"})
else:
    print("Run the Freeze Spec cell first before compiling the construction graph.")

{'api_spec': {'required_endpoints': ['/api/health',
                                     '/api/feedback',
                                     '/api/summary'],
              'required_fields': ['course_name',
                                  'instructor_name',
                                  'rating',
                                  'category',
                                  'feedback_text'],
              'routes': [{'method': 'GET',
                          'path': '/api/health',
                          'purpose': 'Return a lightweight health response for '
                                     'smoke checks.'},
                         {'method': 'GET',
                          'path': '/api/feedback',
                          'purpose': 'Return stored feedback entries for the '
                                     'dashboard list.'},
                         {'method': 'POST',
                          'path': '/api/feedback',
                          'purpose': 'Persi

# 7. Render Outputs

This phase materialises the validated construction outputs as ready application files, UML source files, rendered diagrams, and image assets.

- It writes `app.py`, `index.html`, `requirements.txt`, `Dockerfile`, and seed data.
- It writes `.puml` files first and then renders `.png` diagrams through PlantUML.
- Image generation prefers DashScope and falls back only when the configured flow allows it.


### 7.1 Render Helpers

These helpers resolve downloadable image URLs and render PlantUML outputs into ready diagram files.


In [51]:
def resolve_image_download_url(image_payload: dict) -> str | None:
    candidates = []
    if isinstance(image_payload, dict):
        output = image_payload.get("output", {})
        if isinstance(output, dict):
            results = output.get("results", [])
            if isinstance(results, list):
                for item in results:
                    if isinstance(item, dict):
                        candidates.extend([item.get("url"), item.get("image_url")])

            choices = output.get("choices", [])
            if isinstance(choices, list):
                for choice in choices:
                    if not isinstance(choice, dict):
                        continue
                    message = choice.get("message", {})
                    if not isinstance(message, dict):
                        continue
                    content_items = message.get("content", [])
                    if isinstance(content_items, list):
                        for item in content_items:
                            if isinstance(item, dict):
                                candidates.extend(
                                    [item.get("image"), item.get("url"), item.get("image_url")]
                                )

        data_items = image_payload.get("data")
        if isinstance(data_items, list):
            for item in data_items:
                if isinstance(item, dict):
                    candidates.extend([item.get("url"), item.get("image_url")])

    for candidate in candidates:
        if isinstance(candidate, str) and candidate.strip():
            return candidate.strip()
    return None


def render_uml_diagrams(uml_source: dict) -> dict:
    use_case_source_path = DESIGN_DIR / "use_case_diagram.puml"
    activity_source_path = DESIGN_DIR / "activity_diagram.puml"
    use_case_image_path = DESIGN_DIR / "use_case_diagram.png"
    activity_image_path = DESIGN_DIR / "activity_diagram.png"

    write_text(use_case_source_path, uml_source["use_case"])
    write_text(activity_source_path, uml_source["activity"])

    # UML images are rendered deterministically so incomplete outputs fail loudly.
    render_plantuml_diagram(uml_source["use_case"], use_case_image_path)
    render_plantuml_diagram(uml_source["activity"], activity_image_path)

    return {
        "use_case_diagram": str(use_case_source_path),
        "activity_diagram": str(activity_source_path),
        "use_case_diagram_png": str(use_case_image_path),
        "activity_diagram_png": str(activity_image_path),
    }

### 7.2 Export Execution

This cell writes all submission artefacts, renders UML files, downloads the generated image, and mirrors it into the exported Flask app.


In [52]:
def render_build_artifacts(state: BuildState, config: dict) -> dict:
    # Copy the generated hero image into the exported app so local Flask runs can serve it directly.
    write_json(SPEC_DIR / "backend_contract.json", state["backend_contract"])
    write_json(SPEC_DIR / "frontend_contract.json", state["frontend_contract"])
    write_json(SPEC_DIR / "uml_contract.json", state["uml_contract"])
    write_json(SPEC_DIR / "api_spec.json", state["api_spec"])
    write_json(SPEC_DIR / "ui_spec.json", state["ui_spec"])
    write_json(SPEC_DIR / "uml_source.json", state["uml_source"])
    write_text(SPEC_DIR / "image_prompt.txt", state["image_prompt"])

    uml_report = render_uml_diagrams(state["uml_source"])

    image_target = ASSETS_DIR / "generated_hero.png"
    app_image_target = APP_DIR / "generated_hero.png"
    image_report = {"status": "skipped", "source": None}
    try:
        image_payload = generate_qwen_image(
            prompt=state["image_prompt"],
            api_key=config["dashscope_api_key"],
            model=config["qwen_image_model"],
            region=config.get("dashscope_region"),
            endpoint=config.get("dashscope_image_endpoint"),
        )
        image_url = resolve_image_download_url(image_payload)
        if not image_url:
            raise ValueError("DashScope image payload did not include a downloadable URL.")
        download_image(image_url, image_target)
        shutil.copyfile(image_target, app_image_target)
        image_report = {
            "status": "generated",
            "source": "dashscope",
            "payload_keys": sorted(image_payload.keys()),
        }
    except Exception as exc:
        download_image(FALLBACK_HERO_IMAGE_URL, image_target)
        shutil.copyfile(image_target, app_image_target)
        image_report = {"status": "fallback", "source": "coresg", "reason": str(exc)}

    build_report = {"image": image_report, "uml": uml_report}
    write_json(SPEC_DIR / "build_report.json", build_report)
    write_json(APP_DIR / "seed_feedback.json", SEED_FEEDBACK)
    write_text(APP_DIR / "app.py", state["backend_code"])
    write_text(APP_DIR / "index.html", state["frontend_code"])
    write_text(APP_DIR / "requirements.txt", REQUIREMENTS_TEMPLATE)
    write_text(APP_DIR / "Dockerfile", DOCKERFILE_TEMPLATE)

    return {
        "backend_contract": str(SPEC_DIR / "backend_contract.json"),
        "frontend_contract": str(SPEC_DIR / "frontend_contract.json"),
        "uml_contract": str(SPEC_DIR / "uml_contract.json"),
        "api_spec": str(SPEC_DIR / "api_spec.json"),
        "ui_spec": str(SPEC_DIR / "ui_spec.json"),
        "uml_source": str(SPEC_DIR / "uml_source.json"),
        "image_prompt": str(SPEC_DIR / "image_prompt.txt"),
        "use_case_diagram": str(DESIGN_DIR / "use_case_diagram.puml"),
        "activity_diagram": str(DESIGN_DIR / "activity_diagram.puml"),
        "use_case_diagram_png": str(DESIGN_DIR / "use_case_diagram.png"),
        "activity_diagram_png": str(DESIGN_DIR / "activity_diagram.png"),
        "generated_hero": str(image_target),
        "app_generated_hero": str(app_image_target),
        "seed_feedback": str(APP_DIR / "seed_feedback.json"),
        "app": str(APP_DIR / "app.py"),
        "index": str(APP_DIR / "index.html"),
        "requirements": str(APP_DIR / "requirements.txt"),
        "dockerfile": str(APP_DIR / "Dockerfile"),
        "image_report": image_report,
        "uml_report": uml_report,
    }


rendered_files = None
if build_state:
    rendered_files = render_build_artifacts(build_state, CONFIG)
    build_state["rendered_files"] = rendered_files
    pprint(rendered_files)
else:
    print("Run the construction graph cell first before rendering the generated artifacts.")

{'activity_diagram': 'C:\\Users\\22306\\Desktop\\2472212-Chang_Xu\\Task1\\artifacts\\design\\activity_diagram.puml',
 'activity_diagram_png': 'C:\\Users\\22306\\Desktop\\2472212-Chang_Xu\\Task1\\artifacts\\design\\activity_diagram.png',
 'api_spec': 'C:\\Users\\22306\\Desktop\\2472212-Chang_Xu\\Task1\\artifacts\\spec\\api_spec.json',
 'app': 'C:\\Users\\22306\\Desktop\\2472212-Chang_Xu\\Task1\\artifacts\\app\\app.py',
 'app_generated_hero': 'C:\\Users\\22306\\Desktop\\2472212-Chang_Xu\\Task1\\artifacts\\app\\generated_hero.png',
 'backend_contract': 'C:\\Users\\22306\\Desktop\\2472212-Chang_Xu\\Task1\\artifacts\\spec\\backend_contract.json',
 'dockerfile': 'C:\\Users\\22306\\Desktop\\2472212-Chang_Xu\\Task1\\artifacts\\app\\Dockerfile',
 'frontend_contract': 'C:\\Users\\22306\\Desktop\\2472212-Chang_Xu\\Task1\\artifacts\\spec\\frontend_contract.json',
 'generated_hero': 'C:\\Users\\22306\\Desktop\\2472212-Chang_Xu\\Task1\\artifacts\\assets\\generated_hero.png',
 'image_prompt': 'C:\\Us

# 8. Validate Outputs

This phase applies deterministic checks to confirm that every required export exists and that any optional render failures remain visible for human review before end.

- Missing required outputs means the pipeline is not ready for end.
- Missing optional outputs usually means an external rendering service needs follow-up through `build_report.json`.


In [53]:
generated_files = [
    SPEC_DIR / "problem_statement.md",
    SPEC_DIR / "personas.json",
    SPEC_DIR / "requirements.json",
    SPEC_DIR / "user_stories.json",
    SPEC_DIR / "frozen_inception_spec.json",
    SPEC_DIR / "backend_contract.json",
    SPEC_DIR / "frontend_contract.json",
    SPEC_DIR / "uml_contract.json",
    SPEC_DIR / "api_spec.json",
    SPEC_DIR / "ui_spec.json",
    SPEC_DIR / "uml_source.json",
    SPEC_DIR / "image_prompt.txt",
    DESIGN_DIR / "use_case_diagram.puml",
    DESIGN_DIR / "activity_diagram.puml",
    ASSETS_DIR / "generated_hero.png",
    APP_DIR / "seed_feedback.json",
    APP_DIR / "app.py",
    APP_DIR / "index.html",
    APP_DIR / "requirements.txt",
    APP_DIR / "Dockerfile",
]

optional_generated_files = [
    DESIGN_DIR / "use_case_diagram.png",
    DESIGN_DIR / "activity_diagram.png",
]

missing_outputs = [
    str(path.relative_to(TASK1_DIR))
    for path in generated_files
    if not path.exists()
]

missing_optional_outputs = [
    str(path.relative_to(TASK1_DIR))
    for path in optional_generated_files
    if not path.exists()
]

validation_summary = {
    "generated_count": len(generated_files) - len(missing_outputs),
    "expected_count": len(generated_files),
    "missing_outputs": missing_outputs,
    "missing_optional_outputs": missing_optional_outputs,
}
validation_summary

{'generated_count': 20,
 'expected_count': 20,
 'missing_outputs': [],
 'missing_optional_outputs': []}

# 9. Export Summary

This final phase prints the generated file inventory so the package can be reviewed quickly against the expected Task1 deliverables.

- Confirm that `frozen_inception_spec.json` is refreshed.
- Confirm that UML source files and rendered PNG files are both present.
- Confirm that the generated website can start locally.


In [54]:
print("Generated files:")
for path in generated_files:
    print(f"- {path.relative_to(TASK1_DIR)}")

if missing_outputs:
    print("\nMissing outputs:")
    for missing in missing_outputs:
        print(f"- {missing}")
else:
    print("\nAll expected outputs are present.")

Generated files:
- artifacts\spec\problem_statement.md
- artifacts\spec\personas.json
- artifacts\spec\requirements.json
- artifacts\spec\user_stories.json
- artifacts\spec\frozen_inception_spec.json
- artifacts\spec\backend_contract.json
- artifacts\spec\frontend_contract.json
- artifacts\spec\uml_contract.json
- artifacts\spec\api_spec.json
- artifacts\spec\ui_spec.json
- artifacts\spec\uml_source.json
- artifacts\spec\image_prompt.txt
- artifacts\design\use_case_diagram.puml
- artifacts\design\activity_diagram.puml
- artifacts\assets\generated_hero.png
- artifacts\app\seed_feedback.json
- artifacts\app\app.py
- artifacts\app\index.html
- artifacts\app\requirements.txt
- artifacts\app\Dockerfile

All expected outputs are present.
